In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns
from google.genai.local_tokenizer import LocalTokenizer
from transformers import AutoTokenizer

In [ ]:
plt.rcParams.update(
    {
        "font.family": "serif",
        "font.serif": ["Times New Roman", "DejaVu Serif"],  # Falls back to DejaVu if Times isn't installed
        "axes.labelsize": 10,
        "font.size": 10,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "figure.figsize": (5.5, 3.5),
        "figure.dpi": 300,
    }
)

In [ ]:
DATA_DIR = Path("../../data/")
STRUCTURED_MODEL = Path("../../outputs/training/structured/pretraining/20260506_164441/final_model")
NL_MODEL = Path("../../pre_trained_models/Qwen3-0.6B")

In [ ]:
structured_dir = DATA_DIR / "structured" / "prompts"
split = "train"

dfs = []
for file in structured_dir.glob(f"*/{split}/*.parquet"):
    print(file)
    df = pl.read_parquet(file)
    task = (file.parent.parent.name).replace("_", " ").title()
    df = df.with_columns(pl.lit(task).alias("task"), pl.lit("Structured").alias("method"))
    dfs.append(df)
structured_df = pl.concat(dfs)
dfs = None
structured_df

In [ ]:
natural_language_dir = DATA_DIR / "natural_language" / "raw"
split = "train"

dfs = []
for file in natural_language_dir.glob(f"*/{split}/*.parquet"):
    print(file)
    df = pl.read_parquet(file)
    task = (file.parent.parent.name).replace("_", " ").title()
    df = df.with_columns(pl.lit(task).alias("task"))
    dfs.append(df)
natural_language_df = pl.concat(dfs)
natural_language_df

In [ ]:
qwen_df = natural_language_df.with_columns(pl.lit("Qwen").alias("method"))
gemini_df = natural_language_df.with_columns(pl.lit("Gemini").alias("method"))

In [ ]:
df = pl.concat([structured_df, qwen_df, gemini_df])
structured_df = None
qwen_df = None
gemini_df = None

In [ ]:
structured_tokenizer = AutoTokenizer.from_pretrained(STRUCTURED_MODEL)
nl_tokenizer = AutoTokenizer.from_pretrained(NL_MODEL)
gemini_tokenizer = LocalTokenizer("gemini-3-pro-preview")


def count_tokens(text, method):
    if method == "Structured":
        return len(structured_tokenizer.encode(text))
    elif method == "Qwen":
        return len(nl_tokenizer.encode(text))
    elif method == "Gemini":
        return gemini_tokenizer.count_tokens(text).total_tokens
    else:
        raise ValueError(f"Unknown method: {method}")

In [ ]:
# TODO: Will have to do this better on full dataset, but for now this is fine
df = df.with_columns(
    pl.struct(["ehr_text", "method"])
    .map_elements(lambda x: count_tokens(x["ehr_text"], x["method"]), return_dtype=pl.Int32)
    .alias("token_count")
)
df

In [ ]:
plt.figure()

ax = sns.boxplot(
    data=df,
    x="token_count",
    y="task",
    hue="method",
    palette=["#377eb8", "#ff7f00", "#4daf4a"],
    width=0.6,
    linewidth=1.2,
    fliersize=4,
    flierprops={"alpha": 0.4, "markeredgecolor": "black"},
)

ax.xaxis.grid(True, linestyle="--", alpha=0.6)
ax.set_axisbelow(True)

plt.ylabel("Task")
plt.xlabel("Token Count")
plt.legend(title="Method", title_fontsize="8", fontsize="7")

sns.despine()

plt.tight_layout()
plt.show()

In [ ]:
summary_df = df.group_by("method").agg(
    pl.col("token_count").quantile(0.25).cast(pl.Int32).alias("p25"),
    pl.col("token_count").quantile(0.50).cast(pl.Int32).alias("p50"),
    pl.col("token_count").quantile(0.75).cast(pl.Int32).alias("p75"),
    pl.col("token_count").quantile(0.99).cast(pl.Int32).alias("p99"),
)
summary_df